# 02 — Price Scenario Validation

Visual validation of Schwartz-Smith oil and gas price scenarios.
Checks fan charts, correlation structure, and summary statistics.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt

from src.config import load_config
from src.price import generate_price_scenarios

# master_seed=8 matches the median-HVI experiment run.
cfg = load_config('../config/default.yaml', overrides={'master_seed': 8})
prices = generate_price_scenarios(cfg)

years = np.arange(cfg['n_years'])
print(f'Generated {prices.n_scenarios} scenarios, {prices.n_years} years')


## Oil Price Fan Chart (P10 / P50 / P90)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

p10 = np.percentile(prices.oil, 10, axis=0)
p50 = np.percentile(prices.oil, 50, axis=0)
p90 = np.percentile(prices.oil, 90, axis=0)

ax.fill_between(years, p10, p90, alpha=0.2, color='green', label='P10–P90')
ax.plot(years, p50, color='green', linewidth=2, label='P50 (median)')
ax.plot(years, prices.oil.mean(axis=0), '--', color='darkgreen', label='Mean')

# Plot a few sample paths
for i in range(5):
    ax.plot(years, prices.oil[i], color='gray', alpha=0.3, linewidth=0.5)

ax.set_xlabel('Year')
ax.set_ylabel('Oil Price (USD/bbl)')
ax.set_title('Oil Price Scenarios — Schwartz-Smith')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(bottom=0)
plt.tight_layout()
plt.show()

## Gas Price Fan Chart (P10 / P50 / P90)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

p10 = np.percentile(prices.gas, 10, axis=0)
p50 = np.percentile(prices.gas, 50, axis=0)
p90 = np.percentile(prices.gas, 90, axis=0)

ax.fill_between(years, p10, p90, alpha=0.2, color='blue', label='P10–P90')
ax.plot(years, p50, color='blue', linewidth=2, label='P50 (median)')
ax.plot(years, prices.gas.mean(axis=0), '--', color='darkblue', label='Mean')

for i in range(5):
    ax.plot(years, prices.gas[i], color='gray', alpha=0.3, linewidth=0.5)

ax.set_xlabel('Year')
ax.set_ylabel('Gas Price (USD/MMBtu)')
ax.set_title('Gas Price Scenarios — Schwartz-Smith')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(bottom=0)
plt.tight_layout()
plt.show()

## Combined Oil and Gas Fan Chart (no sample paths)

In [ ]:
fig, (ax_oil, ax_gas) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

# Oil panel
p10 = np.percentile(prices.oil, 10, axis=0)
p50 = np.percentile(prices.oil, 50, axis=0)
p90 = np.percentile(prices.oil, 90, axis=0)
ax_oil.fill_between(years, p10, p90, alpha=0.25, color='green', label='P10–P90')
ax_oil.plot(years, p50, color='green', linewidth=2.5, label='P50 (median)')
ax_oil.plot(years, prices.oil.mean(axis=0), '--', color='darkgreen', linewidth=2, label='Mean')
ax_oil.set_ylabel('Oil Price (USD/bbl)', fontsize=12)
ax_oil.set_title('Oil', fontsize=13)
ax_oil.legend(fontsize=11, loc='best')
ax_oil.grid(True, alpha=0.3)
ax_oil.set_ylim(bottom=0)
ax_oil.tick_params(labelsize=11)

# Gas panel
p10 = np.percentile(prices.gas, 10, axis=0)
p50 = np.percentile(prices.gas, 50, axis=0)
p90 = np.percentile(prices.gas, 90, axis=0)
ax_gas.fill_between(years, p10, p90, alpha=0.25, color='steelblue', label='P10–P90')
ax_gas.plot(years, p50, color='steelblue', linewidth=2.5, label='P50 (median)')
ax_gas.plot(years, prices.gas.mean(axis=0), '--', color='darkblue', linewidth=2, label='Mean')
ax_gas.set_xlabel('Year', fontsize=12)
ax_gas.set_ylabel('Gas Price (USD/MMBtu)', fontsize=12)
ax_gas.set_title('Gas', fontsize=13)
ax_gas.legend(fontsize=11, loc='best')
ax_gas.grid(True, alpha=0.3)
ax_gas.set_ylim(bottom=0)
ax_gas.tick_params(labelsize=11)

# Match x-axis exactly to data range (removes matplotlib's auto padding)
ax_gas.set_xlim(0, cfg['n_years'] - 1)

plt.tight_layout()
plt.show()

## Oil-Gas Correlation

The cross-commodity correlation (rho_oil_gas=0.64) acts on short-term shocks only.
Since short-term components mean-revert, observed price correlation decays over time.

In [ ]:
# Correlation over time (skip year 0 — all scenarios have identical prices there)
correlations = []
for y in range(cfg['n_years']):
    if prices.oil[:, y].std() < 1e-10 or prices.gas[:, y].std() < 1e-10:
        correlations.append(np.nan)
    else:
        correlations.append(np.corrcoef(prices.oil[:, y], prices.gas[:, y])[0, 1])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: correlation over time
axes[0].plot(years, correlations, 'o-', color='purple')
axes[0].axhline(y=cfg['price']['rho_oil_gas'], linestyle='--', color='gray',
                label=f'rho_oil_gas = {cfg["price"]["rho_oil_gas"]} (shock level)')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Price Correlation')
axes[0].set_title('Oil-Gas Price Correlation Over Time')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Right: scatter at midpoint year
scatter_year = cfg['n_years'] // 3
axes[1].scatter(prices.oil[:, scatter_year], prices.gas[:, scatter_year], alpha=0.3, s=5)
axes[1].set_xlabel(f'Oil Price Year {scatter_year} (USD/bbl)')
axes[1].set_ylabel(f'Gas Price Year {scatter_year} (USD/MMBtu)')
corr_y = correlations[scatter_year]
axes[1].set_title(f'Oil vs Gas at Year {scatter_year} (corr={corr_y:.3f})')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Summary Statistics

In [ ]:
# Summary years: evenly spaced across the horizon
n_y = cfg['n_years']
summary_years = [0, n_y // 3, 2 * n_y // 3, n_y - 1]

print('=== Oil Price (USD/bbl) ===')
print(f'{"Year":>6} {"Mean":>8} {"Std":>8} {"Min":>8} {"P10":>8} {"P90":>8} {"Max":>8}')
print('-' * 54)
for y in summary_years:
    d = prices.oil[:, y]
    print(f'{y:>6} {d.mean():>8.1f} {d.std():>8.1f} {d.min():>8.1f} '
          f'{np.percentile(d, 10):>8.1f} {np.percentile(d, 90):>8.1f} {d.max():>8.1f}')

print(f'\n=== Gas Price (USD/MMBtu) ===')
print(f'{"Year":>6} {"Mean":>8} {"Std":>8} {"Min":>8} {"P10":>8} {"P90":>8} {"Max":>8}')
print('-' * 54)
for y in summary_years:
    d = prices.gas[:, y]
    print(f'{y:>6} {d.mean():>8.2f} {d.std():>8.2f} {d.min():>8.2f} '
          f'{np.percentile(d, 10):>8.2f} {np.percentile(d, 90):>8.2f} {d.max():>8.2f}')